# Demonstração PySpark no AWS Glue Studio — laboratório fictício

**Objetivo:** apresentar raciocínio de Engenharia de Dados: contrato, Bronze, validação, quarentena, Silver determinística, Gold e reconciliação.

**Sem dados reais de Banco Inter, Reply ou clientes.** O notebook usa exclusivamente CSV sintético deste repositório. AWS Glue Studio deve estar na região **us-east-2 (Ohio)** e com IAM role autorizada a ler e gravar no bucket. A sessão interativa é cobrada.

**Pré-requisito:** fazer upload de `interview_lab/data/transactions.csv` para `s3://aws-lakehouse-case-lab-2026/interview-lab/input/transactions.csv`. Use os comandos do guia `docs/AWS_DEMO.md`.

Em Glue Studio: **Jobs → Jupyter notebook → Upload and edit an existing notebook**, selecione este arquivo `.ipynb`. Configure Glue 5.0 (Spark 3.5), role e número mínimo de workers permitido. Leia os custos e encerre a sessão no final.

In [ ]:
from datetime import datetime, timezone
import json
import boto3
from pyspark.sql import SparkSession, Window, functions as F, types as T

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version)

## 1. Configuração segura

Somente fontes sintéticas. Saídas usam um **RUN_ID novo**, evitando sobrescrever evidências antigas. Reexecute esta célula para gerar um novo RUN_ID antes de gravar novamente.

In [ ]:
AWS_REGION = "us-east-2"
BUCKET = "aws-lakehouse-case-lab-2026"  # Ajuste se o bucket for diferente.
RUN_ID = "notebook-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
INPUT_URI = f"s3://{BUCKET}/interview-lab/input/transactions.csv"
BASE_URI = f"s3://{BUCKET}/interview-lab/runs/{RUN_ID}"
print("Origem:", INPUT_URI)
print("Destino:", BASE_URI)

## 2. Bronze — schema explícito e preservação do evento

Tipar campos monetários só após receber o valor original permite registrar erros e rastrear os dados.

In [ ]:
raw_schema = T.StructType([
    T.StructField("event_id", T.StringType()),
    T.StructField("transaction_id", T.StringType()),
    T.StructField("account_id", T.StringType()),
    T.StructField("amount_str", T.StringType()),
    T.StructField("event_ts_str", T.StringType()),
    T.StructField("ingested_at_str", T.StringType()),
    T.StructField("status", T.StringType()),
    T.StructField("_corrupt_record", T.StringType()),
])

bronze = (
    spark.read.option("header", "true")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(raw_schema)
    .csv(INPUT_URI)
    .withColumn("source_file", F.input_file_name())
)
print("Registros Bronze:", bronze.count())
bronze.select(
    "event_id", "transaction_id", "account_id",
    "amount_str", "event_ts_str", "status"
).show(20, truncate=False)

## 3. Contrato de qualidade e quarentena

Regra **didática**: valores devem ser positivos; valores negativos podem ser legítimos em sistemas financeiros reais e requerem semântica definida pelo negócio. `decimal(18,2)` preserva precisão decimal. `try_cast` e `try_to_timestamp` sinalizam conversões inválidas.

In [ ]:
prepared = (
    bronze
    .withColumn("event_id", F.trim("event_id"))
    .withColumn("transaction_id", F.trim("transaction_id"))
    .withColumn("account_id", F.trim("account_id"))
    .withColumn("status", F.upper(F.trim("status")))
    .withColumn("amount_raw", F.trim("amount_str"))
    .withColumn("amount", F.expr("try_cast(amount_str as decimal(18,2))"))
    .withColumn(
        "event_ts",
        F.try_to_timestamp("event_ts_str", F.lit("yyyy-MM-dd HH:mm:ss"))
    )
    .withColumn(
        "ingested_at",
        F.try_to_timestamp("ingested_at_str", F.lit("yyyy-MM-dd HH:mm:ss"))
    )
)

amount_ok = F.col("amount_raw").rlike(r"^[0-9]+(\.[0-9]{1,2})?$")
errors = [
    F.when(F.col("_corrupt_record").isNotNull(), "CSV_CORRUPT"),
    F.when(F.col("event_id").isNull() | (F.col("event_id") == ""), "EVENT_ID_MISSING"),
    F.when(F.col("transaction_id").isNull() | (F.col("transaction_id") == ""), "TRANSACTION_ID_MISSING"),
    F.when(F.col("account_id").isNull() | (F.col("account_id") == ""), "ACCOUNT_ID_MISSING"),
    F.when(~amount_ok | F.col("amount").isNull() | (F.col("amount") <= 0), "INVALID_AMOUNT"),
    F.when(F.col("event_ts").isNull(), "INVALID_EVENT_TIME"),
    F.when(F.col("ingested_at").isNull(), "INVALID_INGEST_TIME"),
    F.when(F.col("status").isNull() | ~F.col("status").isin("APPROVED", "PENDING", "CANCELLED"), "INVALID_STATUS"),
]

classified = (
    prepared.withColumn("error_reasons", F.concat_ws("|", *errors))
    .withColumn("is_valid", F.col("error_reasons") == "")
)
valid = classified.filter(F.col("is_valid"))
quarantine = classified.filter(~F.col("is_valid"))
print("Válidos:", valid.count(), "| Quarentena:", quarantine.count())
quarantine.select("event_id", "transaction_id", "error_reasons").show(truncate=False)

## 4. Silver — deduplicação **determinística**

`dropDuplicates(["transaction_id"])` sozinho não escolhe a versão mais recente. A janela define a precedência: horário do evento, horário de ingestão, identificador do evento e hash do conteúdo como último desempate **técnico**. Em produção, uma versão sequencial da origem é preferível ao hash quando existir.

In [ ]:
hashed = valid.withColumn(
    "_payload_hash",
    F.sha2(F.to_json(F.struct(
        "event_id", "transaction_id", "account_id",
        "amount", "event_ts", "ingested_at", "status"
    )), 256)
)
window = Window.partitionBy("transaction_id").orderBy(
    F.col("event_ts").desc(),
    F.col("ingested_at").desc(),
    F.col("event_id").desc(),
    F.col("_payload_hash").desc(),
)
silver = (
    hashed.withColumn("row_num", F.row_number().over(window))
    .filter(F.col("row_num") == 1)
    .drop("row_num", "_payload_hash")
)
silver.select(
    "transaction_id", "account_id", "amount", "event_ts", "status"
).orderBy("transaction_id").show(truncate=False)

## 5. Gold — regra financeira ilustrativa

O total considera **somente o último estado aprovado** de cada transação. Eventos cancelados ou pendentes não são somados.

In [ ]:
gold = (
    silver.filter(F.col("status") == "APPROVED")
    .withColumn("business_date", F.to_date("event_ts"))
    .groupBy("account_id", "business_date")
    .agg(
        F.count("transaction_id").alias("transaction_count"),
        F.sum("amount").cast("decimal(20,2)").alias("total_amount"),
    )
)
gold.orderBy("account_id").show(truncate=False)

## 6. Reconciliação — não aceitar sucesso sem evidências

Reconciliação na origem: Bronze = válidos + quarentena. Após deduplicação: válidos = Silver + eventos superados. Assertions específicas testam o pequeno dataset sintético da demonstração.

In [ ]:
raw_count = bronze.count()
valid_count = valid.count()
quarantine_count = quarantine.count()
silver_count = silver.count()
unique_tx = valid.select("transaction_id").distinct().count()
deduplicated = valid_count - silver_count

assert raw_count == valid_count + quarantine_count, "BRONZE_RECONCILIATION_FAILED"
assert silver_count == unique_tx, "SILVER_RECONCILIATION_FAILED"
assert (raw_count, valid_count, quarantine_count, silver_count, deduplicated) == (10, 7, 3, 5, 2), "SAMPLE_CONTRACT_CHANGED"

gold_totals = {row["account_id"]: str(row["total_amount"]) for row in gold.collect()}
assert gold_totals == {"A100": "150.00", "A300": "40.00"}, gold_totals

report = {
    "run_id": RUN_ID,
    "status": "PASS",
    "runtime": "AWS_GLUE_STUDIO_INTERACTIVE",
    "spark_version": spark.version,
    "raw_count": raw_count,
    "valid_event_count": valid_count,
    "quarantine_count": quarantine_count,
    "silver_count": silver_count,
    "deduplicated_events": deduplicated,
    "unique_transaction_count": unique_tx,
    "gold_totals": gold_totals,
}
print(json.dumps(report, indent=2, ensure_ascii=False))

## 7. Publicar Bronze, Silver, Gold e Quarentena no S3

Todas as gravações usam um prefixo novo `RUN_ID`; o relatório `validation_report.json` é gravado por último. Isso **não torna a escrita transacional/atômica** entre camadas. Em produção, implementar controle de publicação e mecanismo transacional.

In [ ]:
bronze.write.mode("errorifexists").parquet(f"{BASE_URI}/bronze")
silver.write.mode("errorifexists").parquet(f"{BASE_URI}/silver")
quarantine.write.mode("errorifexists").parquet(f"{BASE_URI}/quarantine")
gold.write.mode("errorifexists").partitionBy("business_date").parquet(f"{BASE_URI}/gold")

s3 = boto3.client("s3", region_name=AWS_REGION)
s3.put_object(
    Bucket=BUCKET,
    Key=f"interview-lab/runs/{RUN_ID}/validation_report.json",
    Body=json.dumps(report, ensure_ascii=False, indent=2).encode("utf-8"),
    ContentType="application/json",
)
print("Evidência da execução:", f"{BASE_URI}/validation_report.json")

## 8. Explicar performance e inspecionar o plano

`Window + orderBy` envolve distribuição e ordenação por transação. Em dados grandes, examinar shuffle, skew e custo das ações. O dataset atual é pequeno e **não comprova desempenho em escala**.

In [ ]:
silver.explain("formatted")
print("Demonstração completa:", BASE_URI)

## Encerrar a sessão e evitar cobrança

Ao finalizar a demonstração, em uma nova célula do AWS Glue Studio execute `%stop_session` ou use o botão de encerramento da sessão. **Não execute uma sessão interativa apenas para visualizar o código no GitHub.**

A DAG Airflow/MWAA e o job Glue estão em arquivos separados deste laboratório e exigem implantação e permissões específicas.